In [ ]:
import cudf as cf
import cupy as cp

import optuna
# Use native XGBoost Dask API instead of dask-ml
import xgboost as xgb
# from dask_ml.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error        
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

from plotly.io import show

In [ ]:
df_dropped = cf.read_csv("./tmp/df_39_no_na_dropped.csv")
df_filled = cf.read_csv("./tmp/df_39_no_na_filled.csv")

In [ ]:
df_dropped['delay_seconds'].describe()

count    6.186073e+06
mean    -6.135112e+01
std      7.034961e+02
min     -6.812163e+04
25%     -2.281870e+02
50%     -7.129100e+01
75%      4.800000e+01
max      3.644455e+04
Name: delay_seconds, dtype: float64

In [ ]:
print(f"\nFeature shape(Filled): {df_filled.shape[0]:,} rows x {df_filled.shape[1]} columns")


Feature shape(Filled): 6,186,830 rows x 20 columns


In [ ]:
print(f"\nFeature shape(Dropped): {df_dropped.shape[0]:,} rows x {df_dropped.shape[1]} columns")


Feature shape(Dropped): 6,186,073 rows x 20 columns


## Dropped

In [ ]:
X_train_dropped, X_test_dropped, y_train_dropped, y_test_dropped = train_test_split(
    df_dropped.drop(columns=['delay_seconds', 'trip_id',]),
    df_dropped['delay_seconds'],
    test_size=0.2,
    random_state=42
)

print(f"\nTrain size: {len(X_train_dropped):,}")
print(f"Test size: {len(X_test_dropped):,}")


Train size: 4,948,858
Test size: 1,237,215


In [ ]:
X_train_dropped = X_train_dropped.to_cupy()
y_train_dropped = y_train_dropped.to_cupy()
X_test_dropped = X_test_dropped.to_cupy()
y_test_dropped = y_test_dropped.to_cupy()

In [ ]:
# Define Optuna objective function for hyperparameter tuning
import numpy as np
def objective(trial):
    """Optuna objective function to minimize RMSE using native XGBoost Dask API"""
    
    # Define hyperparameter search space
    params = {
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'device': 'cuda',
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 10_000,15_000, step=1000),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    }
    
    n_samples = X_train_dropped.shape[0]
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    avg_rmse_scores = np.zeros(kf.get_n_splits())
    
    # KFold.split only needs sample count — pass a range, no GPU→CPU data transfer
    for i, (train_index, val_index) in enumerate(kf.split(range(n_samples))):
        # Index cupy arrays directly with numpy indices
        dtrain = xgb.QuantileDMatrix(X_train_dropped[train_index], label=y_train_dropped[train_index])
        dval = xgb.QuantileDMatrix(X_train_dropped[val_index], label=y_train_dropped[val_index], ref=dtrain)

        model = xgb.train(params, dtrain, evals=[(dval, 'eval')], 
                      verbose_eval=False, early_stopping_rounds=100)
    
        # Make predictions (returns numpy array)
        y_pred = model.predict(dval)
    
        # Convert only the fold's labels to numpy for RMSE calculation
        y_fold = y_train_dropped[val_index].get()
        mse = mean_squared_error(y_fold, y_pred)
        rmse = np.sqrt(mse)
        avg_rmse_scores[i] = rmse

    # Return average RMSE across folds
    return float(np.mean(avg_rmse_scores))

print("Objective function defined with native XGBoost and Optuna GPU")

Objective function defined with native XGBoost and Optuna GPU


In [ ]:
# Run Optuna hyperparameter optimization
print("Starting Optuna hyperparameter optimization...")
print("XGBoost will use GPU acceleration, and Optuna will run trials sequentially to maximize GPU utilization.")
print("This may take a while...\n")


# Create study with DaskStorage
study = optuna.create_study(
    direction='minimize',  # Minimize RMSE
    study_name='xgboost_delay_prediction',
    sampler=optuna.samplers.TPESampler(seed=42),
)

# Run optimization sequentially (XGBoost Dask handles parallelization internally)
n_trials = 20  # Total number of trials

print(f"Running {n_trials} trials...")

# Run sequential optimization - each trial uses Dask cluster efficiently
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

print("\n" + "="*60)
print("OPTIMIZATION COMPLETE!")
print("="*60)
print(f"\nBest trial: #{study.best_trial.number}")
print(f"Best RMSE: {study.best_value:.4f}")
print("\nBest hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-02-22 15:58:20,741] A new study created in memory with name: xgboost_delay_prediction


Starting Optuna hyperparameter optimization...
XGBoost will use GPU acceleration, and Optuna will run trials sequentially to maximize GPU utilization.
This may take a while...

Running 20 trials...


  0%|          | 0/20 [00:00<?, ?it/s]

/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [15:58:21] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [15:58:21] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [15:58:22] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [15:58:22] WARNING: /__w/xg

[I 2026-02-22 15:58:22,792] Trial 0 finished with value: 328.6969299872555 and parameters: {'max_depth': 5, 'learning_rate': 0.2536999076681772, 'n_estimators': 14000, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746}. Best is trial 0 with value: 328.6969299872555.


...

[I 2026-02-22 15:58:54,190] Trial 17 finished with value: 444.1182083461466 and parameters: {'max_depth': 6, 'learning_rate': 0.0640593263272859, 'n_estimators': 15000, 'subsample': 0.6799217399729626, 'colsample_bytree': 0.7442968084843167}. Best is trial 14 with value: 309.7009374033745.


/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [15:58:54] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [15:58:54] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [15:58:55] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [15:58:55] WARNING: /__w/xg

[I 2026-02-22 15:58:56,230] Trial 18 finished with value: 301.7329931785642 and parameters: {'max_depth': 8, 'learning_rate': 0.13449894964570103, 'n_estimators': 12000, 'subsample': 0.7318395315629368, 'colsample_bytree': 0.9281889012746768}. Best is trial 18 with value: 301.7329931785642.


/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [15:58:56] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [15:58:56] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [15:58:57] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [15:58:57] WARNING: /__w/xg

[I 2026-02-22 15:58:58,204] Trial 19 finished with value: 409.6261348208985 and parameters: {'max_depth': 8, 'learning_rate': 0.07022704924686567, 'n_estimators': 12000, 'subsample': 0.7495910822966338, 'colsample_bytree': 0.9722990809713067}. Best is trial 18 with value: 301.7329931785642.

OPTIMIZATION COMPLETE!

Best trial: #18
Best RMSE: 301.7330

Best hyperparameters:
  max_depth: 8
  learning_rate: 0.13449894964570103
  n_estimators: 12000
  subsample: 0.7318395315629368
  colsample_bytree: 0.9281889012746768


In [ ]:
study.best_params

{'max_depth': 8,
 'learning_rate': 0.13449894964570103,
 'n_estimators': 12000,
 'subsample': 0.7318395315629368,
 'colsample_bytree': 0.9281889012746768}

In [ ]:
cp.get_default_memory_pool().free_all_blocks()

In [ ]:
# Train final model with best parameters using native XGBoost Dask API
print("Training final model with best hyperparameters...")
print("="*60)

# Get best parameters
best_params = study.best_params.copy()
best_params['objective'] = 'reg:squarederror'
best_params['tree_method'] = 'hist'  # Required for Dask

# Create QuantileDMatrix for training
dtrain = xgb.QuantileDMatrix(X_train_dropped, y_train_dropped)

# Train final model using native Dask XGBoost
output = xgb.train(
    best_params,
    dtrain,
)

Training final model with best hyperparameters...


/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [15:58:58] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:
dtest = xgb.QuantileDMatrix(X_test_dropped)
y_pred_dropped = output.predict(dtest)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test_dropped.get(), y_pred_dropped))
r2 = r2_score(y_test_dropped.get(), y_pred_dropped)
print("Final Model Performance on Test Set:")
print(f"  RMSE: {rmse:.4f}")
print(f"  R²: {r2:.4f}")

Final Model Performance on Test Set:
  RMSE: 305.5008
  R²: 0.8240


## Filled

In [ ]:
cp.get_default_memory_pool().free_all_blocks() 


In [ ]:
X_train_filled, X_test_filled, y_train_filled, y_test_filled = train_test_split(
    df_filled.drop(columns=['delay_seconds', 'trip_id',]),
    df_filled['delay_seconds'],
    test_size=0.2,
    random_state=42
)

In [ ]:
X_train_filled = X_train_filled.to_cupy()
y_train_filled = y_train_filled.to_cupy()
X_test_filled = X_test_filled.to_cupy()
y_test_filled = y_test_filled.to_cupy()

In [ ]:
# Define Optuna objective function for hyperparameter tuning
import numpy as np
def objective(trial):
    """Optuna objective function to minimize RMSE using native XGBoost Dask API"""
    
    # Define hyperparameter search space
    params = {
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'device': 'cuda',
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 10_000,15_000, step=1000),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    }
    
    n_samples = X_train_filled.shape[0]
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
        
    avg_rmse_scores = np.zeros(kf.get_n_splits())
    
    # KFold.split only needs sample count — pass a range, no GPU→CPU data transfer
    for i, (train_index, val_index) in enumerate(kf.split(range(n_samples))):
        # Index cupy arrays directly with numpy indices
        dtrain = xgb.QuantileDMatrix(X_train_filled[train_index], label=y_train_filled[train_index])
        dval = xgb.QuantileDMatrix(X_train_filled[val_index], label=y_train_filled[val_index], ref=dtrain)

        model = xgb.train(params, dtrain, evals=[(dval, 'eval')], 
                      verbose_eval=False, early_stopping_rounds=100)
    
        # Make predictions (returns numpy array)
        y_pred = model.predict(dval)
    
        # Convert only the fold's labels to numpy for RMSE calculation
        y_fold = y_train_filled[val_index].get()
        mse = mean_squared_error(y_fold, y_pred)
        rmse = np.sqrt(mse)
        avg_rmse_scores[i] = rmse

    # Return average RMSE across folds
    return float(np.mean(avg_rmse_scores))

print("Objective function defined with native XGBoost and Optuna GPU")

Objective function defined with native XGBoost and Optuna GPU


In [ ]:
# Run Optuna hyperparameter optimization
print("Starting Optuna hyperparameter optimization...")
print("XGBoost will use GPU acceleration, and Optuna will run trials sequentially to maximize GPU utilization.")
print("This may take a while...\n")


# Create study with DaskStorage
study = optuna.create_study(
    direction='minimize',  # Minimize RMSE
    study_name='xgboost_delay_prediction',
    sampler=optuna.samplers.TPESampler(seed=42),
)

# Run optimization sequentially (XGBoost Dask handles parallelization internally)
n_trials = 20  # Total number of trials

print(f"Running {n_trials} trials...")

# Run sequential optimization - each trial uses Dask cluster efficiently
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

print("\n" + "="*60)
print("OPTIMIZATION COMPLETE!")
print("="*60)
print(f"\nBest trial: #{study.best_trial.number}")
print(f"Best RMSE: {study.best_value:.4f}")
print("\nBest hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-02-22 16:02:20,209] A new study created in memory with name: xgboost_delay_prediction


Starting Optuna hyperparameter optimization...
XGBoost will use GPU acceleration, and Optuna will run trials sequentially to maximize GPU utilization.
This may take a while...

Running 20 trials...


  0%|          | 0/20 [00:00<?, ?it/s]

/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [16:02:20] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [16:02:20] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [16:02:21] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [16:02:21] WARNING: /__w/xg

[I 2026-02-22 16:02:21,859] Trial 0 finished with value: 319.3211021902616 and parameters: {'max_depth': 5, 'learning_rate': 0.2536999076681772, 'n_estimators': 14000, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746}. Best is trial 0 with value: 319.3211021902616.


...

[I 2026-02-22 16:02:53,606] Trial 17 finished with value: 294.066263023266 and parameters: {'max_depth': 6, 'learning_rate': 0.1901474395461428, 'n_estimators': 15000, 'subsample': 0.7800911589038579, 'colsample_bytree': 0.8225056124157852}. Best is trial 17 with value: 294.066263023266.


/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [16:02:53] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [16:02:54] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [16:02:54] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [16:02:54] WARNING: /__w/xg

[I 2026-02-22 16:02:55,460] Trial 18 finished with value: 280.119091633401 and parameters: {'max_depth': 8, 'learning_rate': 0.16358474819477517, 'n_estimators': 15000, 'subsample': 0.7599922797534859, 'colsample_bytree': 0.829091815760703}. Best is trial 18 with value: 280.119091633401.


/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [16:02:55] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [16:02:56] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [16:02:56] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [16:02:56] WARNING: /__w/xg

[I 2026-02-22 16:02:57,359] Trial 19 finished with value: 461.18024943260605 and parameters: {'max_depth': 8, 'learning_rate': 0.05294170628099121, 'n_estimators': 12000, 'subsample': 0.7634572409067748, 'colsample_bytree': 0.9337661640656951}. Best is trial 18 with value: 280.119091633401.

OPTIMIZATION COMPLETE!

Best trial: #18
Best RMSE: 280.1191

Best hyperparameters:
  max_depth: 8
  learning_rate: 0.16358474819477517
  n_estimators: 15000
  subsample: 0.7599922797534859
  colsample_bytree: 0.829091815760703


In [ ]:
study.best_params

{'max_depth': 8,
 'learning_rate': 0.16358474819477517,
 'n_estimators': 15000,
 'subsample': 0.7599922797534859,
 'colsample_bytree': 0.829091815760703}

In [ ]:
cp.get_default_memory_pool().free_all_blocks()

In [ ]:
# Train final model with best parameters using native XGBoost Dask API
print("Training final model with best hyperparameters...")
print("="*60)

# Get best parameters
best_params = study.best_params.copy()
best_params['objective'] = 'reg:squarederror'
best_params['tree_method'] = 'hist'  # Required for Dask

# Create QuantileDMatrix for training
dtrain = xgb.QuantileDMatrix(X_train_filled, y_train_filled)

# Train final model using native Dask XGBoost
output = xgb.train(
    best_params,
    dtrain,
)

Training final model with best hyperparameters...


/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [16:02:59] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimators" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:
dtest = xgb.QuantileDMatrix(X_test_filled)
y_pred_filled = output.predict(dtest)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test_filled.get(), y_pred_filled))
r2 = r2_score(y_test_filled.get(), y_pred_filled)
print("Final Model Performance on Test Set:")
print(f"  RMSE: {rmse:.4f}")
print(f"  R²: {r2:.4f}")

Final Model Performance on Test Set:
  RMSE: 279.8791
  R²: 0.8516
